In [39]:
# ============================================================
# MASTER ALL-MRT ROUTE + MAPILLARY PREPROCESSOR
#
# ONE-TIME OFFLINE BUILD
#
# PURPOSE:
#
# 1. Load every MRT used by the demographic SOM
# 2. Find one representative family-support toilet
# 3. Calculate one walking route from MRT → toilet
# 4. Sample checkpoints along the walking route
# 5. Crawl Mapillary around those checkpoints
# 6. Store everything permanently
#
# OUTPUT:
#
# data/generated/mrt_route_library.json
# data/generated/mrt_route_coverage.csv
# data/generated/singapore_osm_toilets.json
# data/generated/route_library/*.json
#
# NO FLASK SERVER.
# NO FRONTEND TRIGGER.
# ============================================================

from pathlib import Path
from collections import defaultdict
from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)

import getpass
import hashlib
import json
import math
import os
import time

import numpy as np
import pandas as pd
import requests


# ============================================================
# PROJECT
# ============================================================

PROJECT_DIR = (
    Path.home()
    / "OneDrive"
    / "Desktop"
    / "family-toilet-search"
)


GENERATED_DIR = (
    PROJECT_DIR
    / "data"
    / "generated"
)


MRT_CONTEXT_PATH = (
    GENERATED_DIR
    / "mrt_context_som.json"
)


MRT_EXITS_PATH = (
    PROJECT_DIR
    / "mrt_exits.geojson"
)


TOILET_CACHE_PATH = (
    GENERATED_DIR
    / "singapore_osm_toilets.json"
)


ROUTE_LIBRARY_DIR = (
    GENERATED_DIR
    / "route_library"
)


MASTER_ROUTE_LIBRARY_PATH = (
    GENERATED_DIR
    / "mrt_route_library.json"
)


ROUTE_COVERAGE_CSV_PATH = (
    GENERATED_DIR
    / "mrt_route_coverage.csv"
)


ROUTE_FAILURE_PATH = (
    GENERATED_DIR
    / "mrt_route_failures.json"
)


for folder in [
    GENERATED_DIR,
    ROUTE_LIBRARY_DIR
]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )


print("Project:", PROJECT_DIR)
print("MRT contexts:", MRT_CONTEXT_PATH)
print("MRT exits:", MRT_EXITS_PATH)
print("Route library:", ROUTE_LIBRARY_DIR)

Project: C:\Users\MSI\OneDrive\Desktop\family-toilet-search
MRT contexts: C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\mrt_context_som.json
MRT exits: C:\Users\MSI\OneDrive\Desktop\family-toilet-search\mrt_exits.geojson
Route library: C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\route_library


In [40]:
# ============================================================
# CONFIGURATION
# ============================================================

PIPELINE_VERSION = (
    "all-mrt-family-route-v1"
)


# ============================================================
# FAMILY-TOILET SEARCH
# ============================================================

MAX_TOILET_RADIUS_M = 2500


# Strong ranking preference for family-support facilities.

CHANGING_TABLE_BONUS_M = 350

WHEELCHAIR_BONUS_M = 180

BABY_SUPPORT_BONUS_M = 120


# ============================================================
# WALKING ROUTES
# ============================================================

ROUTING_URL = (
    "https://routing.openstreetmap.de/"
    "routed-foot/route/v1/driving"
)


ROUTING_WORKERS = 4

ROUTING_RETRIES = 3


# If foot routing fails after retries,
# retain a flagged direct route instead of losing the MRT.

ALLOW_APPROXIMATE_ROUTE_FALLBACK = True


# ============================================================
# ROUTE SAMPLING
# ============================================================

CHECKPOINT_SPACING_M = 80


# ============================================================
# MAPILLARY
# ============================================================

MAPILLARY_URL = (
    "https://graph.mapillary.com/images"
)


IMAGES_PER_CHECKPOINT = 2


NORMAL_IMAGE_DISTANCE_M = 140


EXPANDED_IMAGE_DISTANCE_M = 300


MAPILLARY_BBOX_BUFFER_M = 160


MAPILLARY_CHUNK_SIZE = 8


MAPILLARY_LIMIT = 2000


MAPILLARY_RETRIES = 4


# ============================================================
# REBUILD SETTINGS
# ============================================================

FORCE_REFRESH_TOILETS = False

FORCE_REBUILD_ROUTES = False

FORCE_RECRAWL_MAPILLARY = False


print("Pipeline:", PIPELINE_VERSION)
print("Checkpoint spacing:", CHECKPOINT_SPACING_M, "m")
print("Images/checkpoint:", IMAGES_PER_CHECKPOINT)

Pipeline: all-mrt-family-route-v1
Checkpoint spacing: 80 m
Images/checkpoint: 2


In [41]:
# ============================================================
# MAPILLARY TOKEN
# ============================================================

MAPILLARY_TOKEN = (
    os.environ.get(
        "MAPILLARY_TOKEN"
    )
)


if not MAPILLARY_TOKEN:

    MAPILLARY_TOKEN = getpass.getpass(
        "Mapillary access token: "
    )


if not MAPILLARY_TOKEN:

    raise RuntimeError(
        "A Mapillary token is required."
    )


print(
    "Mapillary token loaded."
)

Mapillary access token:  ········


Mapillary token loaded.


In [42]:
# ============================================================
# JSON + NAME HELPERS
# ============================================================

def load_json(
    path,
    fallback=None
):

    if not path.exists():

        return fallback


    try:

        return json.loads(

            path.read_text(
                encoding="utf-8"
            )

        )

    except Exception as error:

        print(
            "Could not read:",
            path,
            error
        )

        return fallback


def save_json(
    path,
    payload
):

    temporary = (
        path.with_suffix(
            path.suffix
            +
            ".tmp"
        )
    )


    temporary.write_text(

        json.dumps(
            payload,
            indent=2
        ),

        encoding="utf-8"

    )


    temporary.replace(
        path
    )


def normalize_station(
    value
):

    if not value:

        return None


    text = (
        str(value)
        .strip()
        .upper()
    )


    for suffix in [

        " MRT STATION",

        " STATION",

        " MRT"

    ]:

        if text.endswith(
            suffix
        ):

            text = text[
                :
                -len(suffix)
            ].strip()


    return (

        f"{text} MRT"

        if text

        else None

    )


def slugify(
    value
):

    value = (
        str(value)
        .strip()
        .lower()
    )


    characters = []


    previous_dash = False


    for character in value:

        if character.isalnum():

            characters.append(
                character
            )

            previous_dash = False

        else:

            if not previous_dash:

                characters.append(
                    "-"
                )

                previous_dash = True


    return (
        "".join(
            characters
        )
        .strip("-")
    )


def station_slug(
    station
):

    station = (
        normalize_station(
            station
        )
        or
        "unknown-mrt"
    )


    return slugify(

        station.replace(
            " MRT",
            ""
        )

    )


def station_route_path(
    station
):

    return (

        ROUTE_LIBRARY_DIR

        /

        f"{station_slug(station)}.json"

    )

In [43]:
# ============================================================
# GEOGRAPHY
# ============================================================

EARTH_RADIUS_M = 6371000


def haversine_m(
    lat1,
    lon1,
    lat2,
    lon2
):

    lat1 = math.radians(
        float(lat1)
    )

    lon1 = math.radians(
        float(lon1)
    )

    lat2 = math.radians(
        float(lat2)
    )

    lon2 = math.radians(
        float(lon2)
    )


    dlat = (
        lat2
        -
        lat1
    )


    dlon = (
        lon2
        -
        lon1
    )


    a = (

        math.sin(
            dlat / 2
        ) ** 2

        +

        math.cos(
            lat1
        )

        *

        math.cos(
            lat2
        )

        *

        math.sin(
            dlon / 2
        ) ** 2

    )


    return (

        EARTH_RADIUS_M

        *

        2

        *

        math.atan2(

            math.sqrt(a),

            math.sqrt(
                1 - a
            )

        )

    )


def interpolate_point(
    a,
    b,
    fraction
):

    return {

        "lat":

            float(a["lat"])

            +

            (
                float(b["lat"])
                -
                float(a["lat"])
            )
            *
            fraction,

        "lon":

            float(a["lon"])

            +

            (
                float(b["lon"])
                -
                float(a["lon"])
            )
            *
            fraction

    }

In [44]:
# ============================================================
# LOAD ALL SOM MRTS
# ============================================================

raw_contexts = load_json(

    MRT_CONTEXT_PATH,

    []

)


if isinstance(
    raw_contexts,
    list
):

    records = raw_contexts

elif isinstance(
    raw_contexts,
    dict
):

    records = (

        raw_contexts.get(
            "records"
        )

        or

        raw_contexts.get(
            "contexts"
        )

        or

        raw_contexts.get(
            "stations"
        )

        or

        raw_contexts.get(
            "data"
        )

        or

        []

    )

else:

    records = []


contexts_by_station = {}


for record in records:

    station = normalize_station(

        record.get(
            "station"
        )

        or

        record.get(
            "name"
        )

    )


    if not station:

        continue


    contexts_by_station[
        station
    ] = {

        **record,

        "station":
            station

    }


contexts = sorted(

    contexts_by_station.values(),

    key=lambda item:
        item["station"]

)


ALL_MRT_STATIONS = [

    record["station"]

    for record
    in contexts

]


print(
    "MRTs to preprocess:",
    len(
        ALL_MRT_STATIONS
    )
)


for index, station in enumerate(
    ALL_MRT_STATIONS,
    start=1
):

    print(
        f"{index:03d}",
        station
    )

MRTs to preprocess: 73
001 ADMIRALTY MRT
002 ALJUNIED MRT
003 ANG MO KIO MRT
004 BEAUTY WORLD MRT
005 BEDOK MRT
006 BEDOK NORTH MRT
007 BEDOK RESERVOIR MRT
008 BISHAN MRT
009 BOON KENG MRT
010 BOON LAY MRT
011 BOTANIC GARDENS MRT
012 BRADDELL MRT
013 BRIGHT HILL MRT
014 BUANGKOK MRT
015 BUGIS MRT
016 BUKIT BATOK MRT
017 BUKIT GOMBAK MRT
018 BUKIT PANJANG MRT
019 CALDECOTT MRT
020 CASHEW MRT
021 CHINATOWN MRT
022 CHOA CHU KANG MRT
023 CLEMENTI MRT
024 COMMONWEALTH MRT
025 DAKOTA MRT
026 EUNOS MRT
027 FARRER ROAD MRT
028 GEYLANG BAHRU MRT
029 HAVELOCK MRT
030 HOUGANG MRT
031 KAKI BUKIT MRT
032 KALLANG MRT
033 KEMBANGAN MRT
034 KHATIB MRT
035 KOVAN MRT
036 LAKESIDE MRT
037 LAVENDER MRT
038 MACPHERSON MRT
039 MARINE PARADE MRT
040 MARINE TERRACE MRT
041 MARSILING MRT
042 MARYMOUNT MRT
043 MATTAR MRT
044 MAXWELL MRT
045 MAYFLOWER MRT
046 MOUNTBATTEN MRT
047 NOVENA MRT
048 PASIR RIS MRT
049 PAYA LEBAR MRT
050 PIONEER MRT
051 POTONG PASIR MRT
052 PUNGGOL MRT
053 QUEENSTOWN MRT
054 REDHILL MRT

In [45]:
# ============================================================
# MRT EXITS
# ============================================================

mrt_geojson = load_json(

    MRT_EXITS_PATH,

    {}

)


EXITS_BY_STATION = defaultdict(
    list
)


for feature in (
    mrt_geojson.get(
        "features",
        []
    )
):

    geometry = (
        feature.get(
            "geometry"
        )
        or
        {}
    )


    if (
        geometry.get(
            "type"
        )
        !=
        "Point"
    ):

        continue


    coordinates = (
        geometry.get(
            "coordinates"
        )
        or
        []
    )


    if len(coordinates) < 2:

        continue


    properties = (
        feature.get(
            "properties"
        )
        or
        {}
    )


    station = normalize_station(

        properties.get(
            "STATION_NA"
        )

        or

        properties.get(
            "STATION_NAME"
        )

        or

        properties.get(
            "station_name"
        )

        or

        properties.get(
            "NAME"
        )

        or

        properties.get(
            "name"
        )

    )


    if not station:

        continue


    EXITS_BY_STATION[
        station
    ].append({

        "lat":
            float(
                coordinates[1]
            ),

        "lon":
            float(
                coordinates[0]
            )

    })


print(
    "Stations with MRT exits:",
    len(
        EXITS_BY_STATION
    )
)

Stations with MRT exits: 190


In [46]:
# ============================================================
# CELL 8
# RESUMABLE MRT-BASED OSM TOILET CRAWL
#
# Instead of:
#   one enormous Singapore-wide Overpass query
#
# We do:
#   6 MRTs per request
#   ↓
#   query toilets within 2.6 km
#   ↓
#   cache result immediately
#   ↓
#   wait before next request
#
# If interrupted / rate-limited:
# rerun this cell and completed MRT groups are reused.
# ============================================================

from pathlib import Path
import json
import time
import requests


# ============================================================
# OVERPASS SETTINGS
# ============================================================

OVERPASS_ENDPOINTS = [

    "https://overpass-api.de/api/interpreter",

    "https://overpass.kumi.systems/api/interpreter",

    "https://overpass.private.coffee/api/interpreter",

]


# Slightly larger than our final 2500 m selection radius.
OVERPASS_SEARCH_RADIUS_M = 2600


# Keep requests deliberately small.
OVERPASS_MRT_CHUNK_SIZE = 6


# Be polite to public Overpass infrastructure.
OVERPASS_PAUSE_SECONDS = 3.0


# Attempts PER chunk.
OVERPASS_MAX_ATTEMPTS = 8


TOILET_PROGRESS_PATH = (
    GENERATED_DIR
    / "singapore_osm_toilets_progress.json"
)


TOILET_FAILURE_PATH = (
    GENERATED_DIR
    / "singapore_osm_toilet_failures.json"
)


OVERPASS_HEADERS = {

    "User-Agent":
        "NUS-FamilyJourneySearch/1.0 "
        "(academic computational-design project)",

    "Accept":
        "application/json"

}


# ============================================================
# LOAD / SAVE HELPERS
# ============================================================

def load_toilet_progress():

    if not TOILET_PROGRESS_PATH.exists():

        return {

            "completedStations":
                [],

            "toilets":
                {},

            "updatedAt":
                None

        }


    try:

        return json.loads(

            TOILET_PROGRESS_PATH.read_text(
                encoding="utf-8"
            )

        )

    except Exception:

        return {

            "completedStations":
                [],

            "toilets":
                {},

            "updatedAt":
                None

        }


def save_toilet_progress(
    progress
):

    progress[
        "updatedAt"
    ] = time.time()


    temporary = (
        TOILET_PROGRESS_PATH
        .with_suffix(
            ".json.tmp"
        )
    )


    temporary.write_text(

        json.dumps(
            progress,
            indent=2
        ),

        encoding="utf-8"

    )


    temporary.replace(
        TOILET_PROGRESS_PATH
    )


# ============================================================
# REPRESENTATIVE MRT POINT
# ============================================================

def representative_mrt_point(
    station
):

    exits = (
        EXITS_BY_STATION.get(
            station,
            []
        )
    )


    if exits:

        # First actual MRT exit is sufficient because the
        # toilet search radius is 2.6 km.
        return {

            "lat":
                float(
                    exits[0]["lat"]
                ),

            "lon":
                float(
                    exits[0]["lon"]
                )

        }


    # Fallback to coordinates stored in demographic context.

    context = (
        contexts_by_station.get(
            station,
            {}
        )
    )


    lat = (
        context.get(
            "lat"
        )
    )


    lon = (
        context.get(
            "lon"
        )
    )


    if (
        lat is None
        or
        lon is None
    ):

        return None


    return {

        "lat":
            float(lat),

        "lon":
            float(lon)

    }


# ============================================================
# BUILD SMALL OVERPASS QUERY
# ============================================================

def build_toilet_query(
    stations
):

    clauses = []


    for station in stations:

        point = (
            representative_mrt_point(
                station
            )
        )


        if not point:

            continue


        lat = point[
            "lat"
        ]


        lon = point[
            "lon"
        ]


        radius = (
            OVERPASS_SEARCH_RADIUS_M
        )


        clauses.extend([

            (
                f'node["amenity"="toilets"]'
                f'(around:{radius},{lat},{lon});'
            ),

            (
                f'way["amenity"="toilets"]'
                f'(around:{radius},{lat},{lon});'
            ),

            (
                f'relation["amenity"="toilets"]'
                f'(around:{radius},{lat},{lon});'
            ),

        ])


    return (

        "[out:json][timeout:90];\n"
        "(\n"
        +
        "\n".join(
            clauses
        )
        +
        "\n);\n"
        "out center tags;"
    )


# ============================================================
# PARSE TOILETS
# ============================================================

def parse_toilet_elements(
    payload
):

    toilets = []


    for element in payload.get(
        "elements",
        []
    ):

        lat = (
            element.get(
                "lat"
            )
        )


        lon = (
            element.get(
                "lon"
            )
        )


        center = (
            element.get(
                "center"
            )
            or
            {}
        )


        if lat is None:

            lat = center.get(
                "lat"
            )


        if lon is None:

            lon = center.get(
                "lon"
            )


        if (
            lat is None
            or
            lon is None
        ):

            continue


        tags = (
            element.get(
                "tags"
            )
            or
            {}
        )


        toilet = {

            "osmType":
                element.get(
                    "type"
                ),

            "osmId":
                element.get(
                    "id"
                ),

            "lat":
                float(
                    lat
                ),

            "lon":
                float(
                    lon
                ),

            "name":

                tags.get(
                    "name"
                )

                or

                tags.get(
                    "operator"
                )

                or

                tags.get(
                    "brand"
                )

                or

                "Public toilet",

            "changingTable":

                tags.get(
                    "changing_table"
                )

                or

                tags.get(
                    "changing_table:adult"
                ),

            "wheelchair":

                tags.get(
                    "wheelchair"
                )

                or

                tags.get(
                    "toilets:wheelchair"
                ),

            "baby":

                tags.get(
                    "baby"
                )

                or

                tags.get(
                    "baby_feeding"
                ),

            "tags":
                tags

        }


        toilets.append(
            toilet
        )


    return toilets


# ============================================================
# REQUEST ONE CHUNK
#
# POST first.
# If server responds 406, try GET.
# If 429, obey Retry-After / exponential wait.
# ============================================================

def request_overpass_chunk(
    stations
):

    query = (
        build_toilet_query(
            stations
        )
    )


    last_error = None


    for attempt in range(
        OVERPASS_MAX_ATTEMPTS
    ):

        endpoint = (

            OVERPASS_ENDPOINTS[
                attempt
                %
                len(
                    OVERPASS_ENDPOINTS
                )
            ]

        )


        print()

        print(
            "  Endpoint:",
            endpoint
        )


        print(
            "  Attempt:",
            attempt + 1,
            "/",
            OVERPASS_MAX_ATTEMPTS
        )


        response = None


        try:

            # ================================================
            # POST
            # ================================================

            response = requests.post(

                endpoint,

                data={
                    "data":
                        query
                },

                headers=
                    OVERPASS_HEADERS,

                timeout=120

            )


            # ================================================
            # 406 FALLBACK → GET
            # ================================================

            if (
                response.status_code
                ==
                406
            ):

                print(
                    "  POST returned 406.",
                    "Trying GET instead..."
                )


                response = requests.get(

                    endpoint,

                    params={
                        "data":
                            query
                    },

                    headers=
                        OVERPASS_HEADERS,

                    timeout=120

                )


            # ================================================
            # RATE LIMITED
            # ================================================

            if (
                response.status_code
                ==
                429
            ):

                retry_after = (
                    response.headers.get(
                        "Retry-After"
                    )
                )


                try:

                    retry_after = float(
                        retry_after
                    )

                except Exception:

                    retry_after = min(

                        90,

                        10
                        *
                        (
                            attempt
                            +
                            1
                        )

                    )


                print(
                    "  Rate limited (429)."
                )


                print(
                    "  Waiting",
                    retry_after,
                    "seconds..."
                )


                time.sleep(
                    retry_after
                )


                continue


            # ================================================
            # OTHER TEMPORARY SERVER FAILURES
            # ================================================

            if (
                response.status_code
                in
                (
                    502,
                    503,
                    504
                )
            ):

                wait_seconds = min(

                    60,

                    5
                    *
                    (
                        attempt
                        +
                        1
                    )

                )


                print(
                    "  Server busy:",
                    response.status_code
                )


                print(
                    "  Waiting",
                    wait_seconds,
                    "seconds..."
                )


                time.sleep(
                    wait_seconds
                )


                continue


            response.raise_for_status()


            payload = (
                response.json()
            )


            toilets = (
                parse_toilet_elements(
                    payload
                )
            )


            print(
                "  Toilets returned:",
                len(
                    toilets
                )
            )


            return toilets


        except Exception as error:

            last_error = error


            print(
                "  Request failed:",
                error
            )


            wait_seconds = min(

                60,

                4
                *
                (
                    attempt
                    +
                    1
                )

            )


            print(
                "  Waiting",
                wait_seconds,
                "seconds..."
            )


            time.sleep(
                wait_seconds
            )


    raise RuntimeError(

        f"Overpass chunk failed after "
        f"{OVERPASS_MAX_ATTEMPTS} attempts: "
        f"{last_error}"

    )


# ============================================================
# CHUNK HELPER
# ============================================================

def chunks(
    values,
    size
):

    for index in range(
        0,
        len(values),
        size
    ):

        yield values[
            index:
            index + size
        ]


# ============================================================
# MASTER RESUMABLE CRAWL
# ============================================================

def fetch_singapore_toilets():

    # --------------------------------------------------------
    # FINAL CACHE
    # --------------------------------------------------------

    if (
        TOILET_CACHE_PATH.exists()
        and
        not FORCE_REFRESH_TOILETS
    ):

        cached = load_json(

            TOILET_CACHE_PATH,

            []

        )


        if cached:

            print(
                "Using completed cached toilet dataset."
            )


            print(
                "Toilets:",
                len(
                    cached
                )
            )


            return cached


    # --------------------------------------------------------
    # RESUME PARTIAL CACHE
    # --------------------------------------------------------

    progress = (
        load_toilet_progress()
    )


    completed_stations = set(

        progress.get(
            "completedStations",
            []
        )

    )


    toilet_dictionary = (

        progress.get(
            "toilets",
            {}
        )

        or

        {}

    )


    remaining = [

        station

        for station
        in ALL_MRT_STATIONS

        if station
        not in completed_stations

    ]


    print()
    print("========================================")
    print("RESUMABLE OSM TOILET CRAWL")
    print("========================================")


    print(
        "Total MRTs:",
        len(
            ALL_MRT_STATIONS
        )
    )


    print(
        "Already completed:",
        len(
            completed_stations
        )
    )


    print(
        "Remaining:",
        len(
            remaining
        )
    )


    if not remaining:

        final_toilets = list(

            toilet_dictionary.values()

        )


        save_json(

            TOILET_CACHE_PATH,

            final_toilets

        )


        return final_toilets


    station_chunks = list(

        chunks(

            remaining,

            OVERPASS_MRT_CHUNK_SIZE

        )

    )


    failures = []


    for chunk_index, station_chunk in enumerate(
        station_chunks,
        start=1
    ):

        print()
        print(
            "----------------------------------------"
        )


        print(
            f"Chunk {chunk_index}/"
            f"{len(station_chunks)}"
        )


        print(
            "MRTs:",
            " · ".join(
                station_chunk
            )
        )


        try:

            toilets = (
                request_overpass_chunk(
                    station_chunk
                )
            )


            # ================================================
            # DEDUPLICATE GLOBALLY
            # ================================================

            for toilet in toilets:

                key = (

                    f"{toilet['osmType']}:"
                    f"{toilet['osmId']}"

                )


                toilet_dictionary[
                    key
                ] = toilet


            # ================================================
            # MARK THESE MRTS COMPLETE
            # ================================================

            for station in station_chunk:

                completed_stations.add(
                    station
                )


            progress = {

                "completedStations":
                    sorted(
                        completed_stations
                    ),

                "toilets":
                    toilet_dictionary,

                "updatedAt":
                    time.time()

            }


            # SAVE AFTER EVERY CHUNK.

            save_toilet_progress(
                progress
            )


            print(
                "  Global unique toilets:",
                len(
                    toilet_dictionary
                )
            )


        except Exception as error:

            print()
            print(
                "  CHUNK FAILED"
            )


            print(
                " ",
                error
            )


            failures.append({

                "stations":
                    station_chunk,

                "error":
                    str(
                        error
                    )

            })


            # IMPORTANT:
            #
            # We continue rather than losing everything.
            #
            # Rerun this cell later and only failed chunks
            # will still be outstanding.


        print(
            "  Cooling down for",
            OVERPASS_PAUSE_SECONDS,
            "seconds..."
        )


        time.sleep(
            OVERPASS_PAUSE_SECONDS
        )


    # ========================================================
    # SAVE FAILURES
    # ========================================================

    save_json(

        TOILET_FAILURE_PATH,

        failures

    )


    # ========================================================
    # DID EVERYTHING COMPLETE?
    # ========================================================

    missing_stations = [

        station

        for station
        in ALL_MRT_STATIONS

        if station
        not in completed_stations

    ]


    if missing_stations:

        print()
        print("========================================")
        print("TOILET CRAWL PARTIALLY COMPLETE")
        print("========================================")


        print(
            "Completed MRTs:",
            len(
                completed_stations
            ),
            "/",
            len(
                ALL_MRT_STATIONS
            )
        )


        print()
        print(
            "Still remaining:"
        )


        for station in missing_stations:

            print(
                " -",
                station
            )


        print()
        print(
            "Progress has been saved."
        )


        print(
            "Wait a few minutes and rerun THIS CELL."
        )


        raise RuntimeError(

            f"{len(missing_stations)} MRTs "
            "still need their OSM toilet query. "
            "Progress is safely cached."

        )


    # ========================================================
    # COMPLETE
    # ========================================================

    final_toilets = list(

        toilet_dictionary.values()

    )


    save_json(

        TOILET_CACHE_PATH,

        final_toilets

    )


    print()
    print("========================================")
    print("OSM TOILET DATASET COMPLETE")
    print("========================================")


    print(
        "MRTs covered:",
        len(
            completed_stations
        )
    )


    print(
        "Unique toilets:",
        len(
            final_toilets
        )
    )


    return final_toilets


# ============================================================
# EXECUTE
# ============================================================

SINGAPORE_TOILETS = (
    fetch_singapore_toilets()
)

Using completed cached toilet dataset.
Toilets: 729


In [47]:
# ============================================================
# REPRESENTATIVE FAMILY-SUPPORT DESTINATION
# ============================================================

def positive_tag(
    value
):

    return (

        str(
            value
            or
            ""
        )
        .strip()
        .lower()

        in {

            "yes",
            "true",
            "1",
            "designated",
            "limited"

        }

    )


def find_best_toilet_for_station(
    station
):

    exits = (
        EXITS_BY_STATION.get(
            station,
            []
        )
    )


    context = (
        contexts_by_station.get(
            station,
            {}
        )
    )


    # --------------------------------------------------------
    # Fallback if exit GeoJSON doesn't contain this station.
    # --------------------------------------------------------

    if not exits:

        lat = context.get(
            "lat"
        )


        lon = context.get(
            "lon"
        )


        if (
            lat is None
            or
            lon is None
        ):

            return None


        exits = [{

            "lat":
                float(lat),

            "lon":
                float(lon)

        }]


    best = None


    for toilet in SINGAPORE_TOILETS:

        # Find nearest station exit to this toilet.

        nearest_exit = min(

            exits,

            key=lambda exit_point:

                haversine_m(

                    exit_point["lat"],
                    exit_point["lon"],

                    toilet["lat"],
                    toilet["lon"]

                )

        )


        distance = haversine_m(

            nearest_exit["lat"],
            nearest_exit["lon"],

            toilet["lat"],
            toilet["lon"]

        )


        if (
            distance
            >
            MAX_TOILET_RADIUS_M
        ):

            continue


        score = (
            distance
        )


        family_features = []


        if positive_tag(

            toilet.get(
                "changingTable"
            )

        ):

            score -= (
                CHANGING_TABLE_BONUS_M
            )


            family_features.append(
                "changing_table"
            )


        if positive_tag(

            toilet.get(
                "wheelchair"
            )

        ):

            score -= (
                WHEELCHAIR_BONUS_M
            )


            family_features.append(
                "wheelchair"
            )


        if positive_tag(

            toilet.get(
                "baby"
            )

        ):

            score -= (
                BABY_SUPPORT_BONUS_M
            )


            family_features.append(
                "baby_support"
            )


        candidate = {

            "score":
                score,

            "straightLineDistance":
                distance,

            "origin":
                nearest_exit,

            "toilet": {

                **toilet,

                "familyFeatures":
                    family_features

            }

        }


        if (
            best is None
            or
            candidate["score"]
            <
            best["score"]
        ):

            best = candidate


    return best

In [48]:
# ============================================================
# PRE-CHECK TOILET COVERAGE
# ============================================================

destination_rows = []


for station in ALL_MRT_STATIONS:

    candidate = (
        find_best_toilet_for_station(
            station
        )
    )


    if candidate:

        destination_rows.append({

            "station":
                station,

            "destination":
                candidate[
                    "toilet"
                ][
                    "name"
                ],

            "straight_line_m":
                round(
                    candidate[
                        "straightLineDistance"
                    ],
                    1
                ),

            "features":
                ", ".join(

                    candidate[
                        "toilet"
                    ][
                        "familyFeatures"
                    ]

                )

        })


    else:

        destination_rows.append({

            "station":
                station,

            "destination":
                None,

            "straight_line_m":
                None,

            "features":
                None

        })


destination_df = pd.DataFrame(
    destination_rows
)


display(
    destination_df
)


print(
    "MRTs with candidate destination:",
    destination_df[
        "destination"
    ].notna().sum(),
    "/",
    len(
        destination_df
    )
)

,station,destination,straight_line_m,features
0,ADMIRALTY MRT,Public toilet,850.5,
1,ALJUNIED MRT,Public toilet,23.5,
2,ANG MO KIO MRT,Public toilet,833.6,wheelchair
3,BEAUTY WORLD MRT,Public toilet,124.2,wheelchair
4,BEDOK MRT,Public toilet,998.5,
...,...,...,...,...
68,UPPER CHANGI MRT,Public toilet,562.7,wheelchair
69,UPPER THOMSON MRT,Public toilet,88.9,
70,WOODLANDS SOUTH MRT,Public toilet,225.3,wheelchair
71,YEW TEE MRT,Public toilet,355.5,


MRTs with candidate destination: 73 / 73


In [49]:
# ============================================================
# FOOT ROUTING
# ============================================================

def request_walking_route(
    origin,
    destination
):

    coordinates = (

        f"{origin['lon']},"
        f"{origin['lat']};"

        f"{destination['lon']},"
        f"{destination['lat']}"

    )


    url = (

        f"{ROUTING_URL}/"
        f"{coordinates}"

    )


    params = {

        "overview":
            "full",

        "geometries":
            "geojson",

        "steps":
            "false"

    }


    last_error = None


    for attempt in range(
        ROUTING_RETRIES
    ):

        try:

            response = requests.get(

                url,

                params=params,

                timeout=45

            )


            response.raise_for_status()


            payload = (
                response.json()
            )


            routes = (
                payload.get(
                    "routes"
                )
                or
                []
            )


            if not routes:

                raise RuntimeError(
                    "Router returned no route."
                )


            result = routes[0]


            coordinates = (

                result[
                    "geometry"
                ][
                    "coordinates"
                ]

            )


            points = [

                {

                    "lat":
                        float(
                            coordinate[1]
                        ),

                    "lon":
                        float(
                            coordinate[0]
                        )

                }

                for coordinate
                in coordinates

            ]


            return {

                "points":
                    points,

                "distance":
                    float(
                        result.get(
                            "distance",
                            0
                        )
                    ),

                "duration":
                    float(
                        result.get(
                            "duration",
                            0
                        )
                    ),

                "routingMode":
                    "walking"

            }


        except Exception as error:

            last_error = error


            time.sleep(
                1
                +
                attempt
            )


    if (
        ALLOW_APPROXIMATE_ROUTE_FALLBACK
    ):

        distance = haversine_m(

            origin["lat"],
            origin["lon"],

            destination["lat"],
            destination["lon"]

        )


        return {

            "points": [

                {
                    "lat":
                        origin["lat"],

                    "lon":
                        origin["lon"]
                },

                {
                    "lat":
                        destination["lat"],

                    "lon":
                        destination["lon"]
                }

            ],

            "distance":
                distance,

            "duration":
                distance
                /
                1.2,

            "routingMode":
                "approximate",

            "routingError":
                str(
                    last_error
                )

        }


    raise RuntimeError(
        str(
            last_error
        )
    )

In [50]:
# ============================================================
# ROUTE SAMPLING
# ============================================================

def sample_route(
    points,
    spacing_m=
        CHECKPOINT_SPACING_M
):

    if (
        not points
        or
        len(points) < 2
    ):

        return []


    cumulative = [
        0.0
    ]


    for index in range(
        1,
        len(points)
    ):

        distance = haversine_m(

            points[
                index - 1
            ]["lat"],

            points[
                index - 1
            ]["lon"],

            points[
                index
            ]["lat"],

            points[
                index
            ]["lon"]

        )


        cumulative.append(

            cumulative[-1]
            +
            distance

        )


    total_distance = (
        cumulative[-1]
    )


    targets = list(

        np.arange(

            0,

            total_distance,

            spacing_m

        )

    )


    if (
        not targets
        or
        abs(
            targets[-1]
            -
            total_distance
        )
        >
        1
    ):

        targets.append(
            total_distance
        )


    sampled = []


    segment_index = 0


    for checkpoint_index, target in enumerate(
        targets
    ):

        while (

            segment_index
            <
            len(
                cumulative
            )
            -
            2

            and

            cumulative[
                segment_index + 1
            ]
            <
            target

        ):

            segment_index += 1


        start_distance = (
            cumulative[
                segment_index
            ]
        )


        end_distance = (
            cumulative[
                segment_index + 1
            ]
        )


        segment_length = (

            end_distance
            -
            start_distance

        )


        if segment_length <= 0:

            fraction = 0

        else:

            fraction = (

                target
                -
                start_distance

            ) / segment_length


        point = interpolate_point(

            points[
                segment_index
            ],

            points[
                segment_index + 1
            ],

            fraction

        )


        sampled.append({

            "checkpointIndex":
                checkpoint_index,

            "distanceFromStart":
                float(
                    target
                ),

            "lat":
                point[
                    "lat"
                ],

            "lon":
                point[
                    "lon"
                ],

            "imageCandidates":
                []

        })


    return sampled

In [51]:
# ============================================================
# BUILD ROUTE SKELETON FOR ONE MRT
# ============================================================

def build_station_route(
    station
):

    output_path = (
        station_route_path(
            station
        )
    )


    # --------------------------------------------------------
    # RESUME
    # --------------------------------------------------------

    if (
        output_path.exists()
        and
        not FORCE_REBUILD_ROUTES
    ):

        cached = load_json(

            output_path,

            None

        )


        if (
            cached
            and
            cached.get(
                "pipelineVersion"
            )
            ==
            PIPELINE_VERSION
            and
            cached.get(
                "routeStatus"
            )
            ==
            "complete"
        ):

            return cached


    candidate = (
        find_best_toilet_for_station(
            station
        )
    )


    if not candidate:

        return {

            "pipelineVersion":
                PIPELINE_VERSION,

            "station":
                station,

            "routeStatus":
                "failed",

            "failureReason":
                "No toilet candidate"

        }


    origin = (
        candidate[
            "origin"
        ]
    )


    toilet = (
        candidate[
            "toilet"
        ]
    )


    walking = (
        request_walking_route(

            origin,

            toilet

        )
    )


    station_code = (

        station_slug(
            station
        )
        .upper()
        .replace(
            "-",
            "_"
        )

    )


    route_id = (

        f"{station_code}"
        "__R01"

    )


    checkpoints = (
        sample_route(

            walking[
                "points"
            ]

        )
    )


    for checkpoint in checkpoints:

        checkpoint[
            "checkpointId"
        ] = (

            f"{route_id}"
            "__C"
            f"{checkpoint['checkpointIndex']:03d}"

        )


    route_record = {

        "pipelineVersion":
            PIPELINE_VERSION,

        "station":
            station,

        "routeId":
            route_id,

        "routeStatus":
            "complete",

        "mapillaryStatus":
            "pending",

        "routingMode":
            walking[
                "routingMode"
            ],

        "origin":
            origin,

        "toilet":
            toilet,

        "straightLineDistance":
            candidate[
                "straightLineDistance"
            ],

        "distance":
            walking[
                "distance"
            ],

        "duration":
            walking[
                "duration"
            ],

        "points":
            walking[
                "points"
            ],

        "checkpoints":
            checkpoints,

        "mapillaryImageCount":
            0,

        "uniqueMapillaryImageCount":
            0

    }


    if (
        "routingError"
        in walking
    ):

        route_record[
            "routingError"
        ] = walking[
            "routingError"
        ]


    save_json(

        output_path,

        route_record

    )


    return route_record

In [52]:
# ============================================================
# BUILD ALL MRT WALKING ROUTES
# ============================================================

print()
print("==========================================")
print("BUILDING ALL MRT WALKING ROUTES")
print("==========================================")
print()


route_results = {}


with ThreadPoolExecutor(
    max_workers=
        ROUTING_WORKERS
) as executor:

    futures = {

        executor.submit(
            build_station_route,
            station
        ):
            station

        for station
        in ALL_MRT_STATIONS

    }


    completed = 0


    for future in as_completed(
        futures
    ):

        station = (
            futures[
                future
            ]
        )


        completed += 1


        try:

            result = (
                future.result()
            )


            route_results[
                station
            ] = result


            print(

                f"{completed:02d}/"
                f"{len(ALL_MRT_STATIONS)}",

                station,

                "→",

                result.get(
                    "routingMode",
                    result.get(
                        "routeStatus"
                    )
                )

            )


        except Exception as error:

            route_results[
                station
            ] = {

                "station":
                    station,

                "routeStatus":
                    "failed",

                "failureReason":
                    str(
                        error
                    )

            }


            print(

                f"{completed:02d}/"
                f"{len(ALL_MRT_STATIONS)}",

                station,

                "→ FAILED:",

                error

            )


BUILDING ALL MRT WALKING ROUTES

01/73 ADMIRALTY MRT → walking
02/73 ALJUNIED MRT → walking
03/73 ANG MO KIO MRT → walking
04/73 BEDOK MRT → walking
05/73 BEAUTY WORLD MRT → walking
06/73 BEDOK NORTH MRT → walking
07/73 BEDOK RESERVOIR MRT → walking
08/73 BISHAN MRT → walking
09/73 BOON LAY MRT → walking
10/73 BOTANIC GARDENS MRT → walking
11/73 BOON KENG MRT → walking
12/73 BRADDELL MRT → walking
13/73 BUGIS MRT → walking
14/73 BRIGHT HILL MRT → approximate
15/73 BUANGKOK MRT → walking
16/73 BUKIT GOMBAK MRT → walking
17/73 BUKIT PANJANG MRT → walking
18/73 BUKIT BATOK MRT → walking
19/73 CASHEW MRT → walking
20/73 CALDECOTT MRT → approximate
21/73 CHOA CHU KANG MRT → walking
22/73 CLEMENTI MRT → walking
23/73 COMMONWEALTH MRT → walking
24/73 DAKOTA MRT → walking
25/73 CHINATOWN MRT → walking
26/73 HAVELOCK MRT → walking
27/73 FARRER ROAD MRT → approximate
28/73 EUNOS MRT → walking
29/73 GEYLANG BAHRU MRT → walking
30/73 KAKI BUKIT MRT → approximate
31/73 HOUGANG MRT → approximate
32

In [53]:
# ============================================================
# ROUTE COVERAGE
# ============================================================

route_status_rows = []


for station in ALL_MRT_STATIONS:

    route = (
        route_results.get(
            station,
            {}
        )
    )


    route_status_rows.append({

        "station":
            station,

        "route_status":
            route.get(
                "routeStatus"
            ),

        "routing_mode":
            route.get(
                "routingMode"
            ),

        "route_distance_m":
            route.get(
                "distance"
            ),

        "checkpoint_count":
            len(
                route.get(
                    "checkpoints",
                    []
                )
            ),

        "destination":
            (
                route.get(
                    "toilet",
                    {}
                )
                .get(
                    "name"
                )
            )

    })


route_status_df = pd.DataFrame(
    route_status_rows
)


display(
    route_status_df
)


print(
    "Successful route skeletons:",
    (
        route_status_df[
            "route_status"
        ]
        ==
        "complete"
    ).sum(),
    "/",
    len(
        route_status_df
    )
)

,station,route_status,routing_mode,route_distance_m,checkpoint_count,destination
0,ADMIRALTY MRT,complete,walking,1153.7,16,Public toilet
1,ALJUNIED MRT,complete,walking,23.2,2,Public toilet
2,ANG MO KIO MRT,complete,walking,1364.3,19,Public toilet
3,BEAUTY WORLD MRT,complete,walking,177.0,4,Public toilet
4,BEDOK MRT,complete,walking,1392.6,19,Public toilet
...,...,...,...,...,...,...
68,UPPER CHANGI MRT,complete,walking,714.2,10,Public toilet
69,UPPER THOMSON MRT,complete,walking,229.3,4,Public toilet
70,WOODLANDS SOUTH MRT,complete,walking,269.0,5,Public toilet
71,YEW TEE MRT,complete,walking,805.8,12,Public toilet


Successful route skeletons: 73 / 73


In [54]:
# ============================================================
# MAPILLARY BBOX HELPERS
# ============================================================

def bbox_for_points(
    points,
    buffer_m
):

    latitudes = [

        float(
            point["lat"]
        )

        for point
        in points

    ]


    longitudes = [

        float(
            point["lon"]
        )

        for point
        in points

    ]


    center_lat = (

        sum(
            latitudes
        )

        /
        len(
            latitudes
        )

    )


    lat_buffer = (

        buffer_m
        /
        111320

    )


    cos_lat = max(

        0.1,

        math.cos(
            math.radians(
                center_lat
            )
        )

    )


    lon_buffer = (

        buffer_m

        /

        (
            111320
            *
            cos_lat
        )

    )


    west = (
        min(longitudes)
        -
        lon_buffer
    )


    east = (
        max(longitudes)
        +
        lon_buffer
    )


    south = (
        min(latitudes)
        -
        lat_buffer
    )


    north = (
        max(latitudes)
        +
        lat_buffer
    )


    return (

        west,
        south,
        east,
        north

    )

In [55]:
# ============================================================
# MAPILLARY API
# ============================================================

MAPILLARY_FIELDS = ",".join([

    "id",
    "thumb_1024_url",
    "computed_geometry",
    "captured_at",
    "compass_angle"

])


def query_mapillary_bbox(
    bbox
):

    west, south, east, north = (
        bbox
    )


    params = {

        "access_token":
            MAPILLARY_TOKEN,

        "fields":
            MAPILLARY_FIELDS,

        "bbox":

            f"{west},"
            f"{south},"
            f"{east},"
            f"{north}",

        "limit":
            MAPILLARY_LIMIT

    }


    last_error = None


    for attempt in range(
        MAPILLARY_RETRIES
    ):

        try:

            response = requests.get(

                MAPILLARY_URL,

                params=params,

                timeout=45

            )


            if (
                response.status_code
                ==
                429
            ):

                wait_seconds = (

                    2
                    ** attempt

                )


                print(
                    "Mapillary rate limit.",
                    "Waiting",
                    wait_seconds,
                    "seconds."
                )


                time.sleep(
                    wait_seconds
                )


                continue


            response.raise_for_status()


            payload = (
                response.json()
            )


            return (
                payload.get(
                    "data"
                )
                or
                []
            )


        except Exception as error:

            last_error = error


            time.sleep(

                1
                +
                attempt

            )


    raise RuntimeError(

        f"Mapillary request failed: "
        f"{last_error}"

    )

In [56]:
# ============================================================
# MAPILLARY CANDIDATE PARSER
# ============================================================

def parse_mapillary_record(
    record
):

    geometry = (

        record.get(
            "computed_geometry"
        )

        or

        {}

    )


    coordinates = (

        geometry.get(
            "coordinates"
        )

        or

        []

    )


    if len(coordinates) < 2:

        return None


    return {

        "id":
            str(
                record[
                    "id"
                ]
            ),

        "thumb_1024_url":
            record.get(
                "thumb_1024_url"
            ),

        "lat":
            float(
                coordinates[1]
            ),

        "lon":
            float(
                coordinates[0]
            ),

        "captured_at":
            record.get(
                "captured_at"
            ),

        "compass_angle":
            record.get(
                "compass_angle"
            )

    }

In [57]:
# ============================================================
# FETCH MAPILLARY ALONG ROUTE
# ============================================================

def chunks(
    values,
    size
):

    for index in range(
        0,
        len(values),
        size
    ):

        yield values[
            index:
            index + size
        ]


def fetch_candidates_for_route(
    checkpoints,
    buffer_m=
        MAPILLARY_BBOX_BUFFER_M
):

    unique = {}


    for checkpoint_group in chunks(

        checkpoints,

        MAPILLARY_CHUNK_SIZE

    ):

        bbox = bbox_for_points(

            checkpoint_group,

            buffer_m

        )


        records = query_mapillary_bbox(
            bbox
        )


        for record in records:

            parsed = (
                parse_mapillary_record(
                    record
                )
            )


            if not parsed:

                continue


            unique[
                parsed["id"]
            ] = parsed


        # Gentle throttling.

        time.sleep(
            0.12
        )


    return list(
        unique.values()
    )

In [58]:
# ============================================================
# ASSIGN IMAGES TO CHECKPOINTS
# ============================================================

def assign_images_to_checkpoints(
    checkpoints,
    candidates,
    maximum_distance_m
):

    if not candidates:

        return checkpoints


    candidate_lats = np.array(

        [
            candidate["lat"]
            for candidate
            in candidates
        ],

        dtype=float

    )


    candidate_lons = np.array(

        [
            candidate["lon"]
            for candidate
            in candidates
        ],

        dtype=float

    )


    for checkpoint in checkpoints:

        lat = float(
            checkpoint["lat"]
        )


        lon = float(
            checkpoint["lon"]
        )


        lat_scale = (
            111320
        )


        lon_scale = (

            111320

            *

            max(

                0.1,

                math.cos(
                    math.radians(
                        lat
                    )
                )

            )

        )


        dy = (

            candidate_lats
            -
            lat

        ) * lat_scale


        dx = (

            candidate_lons
            -
            lon

        ) * lon_scale


        distances = np.sqrt(

            dx ** 2

            +

            dy ** 2

        )


        nearest_indices = np.argsort(

            distances

        )[
            :
            IMAGES_PER_CHECKPOINT
        ]


        image_candidates = []


        for candidate_index in (
            nearest_indices
        ):

            distance = float(

                distances[
                    candidate_index
                ]

            )


            if (
                distance
                >
                maximum_distance_m
            ):

                continue


            candidate = {

                **candidates[
                    int(
                        candidate_index
                    )
                ],

                "distance":
                    distance

            }


            image_candidates.append(
                candidate
            )


        checkpoint[
            "imageCandidates"
        ] = image_candidates


    return checkpoints

In [59]:
# ============================================================
# CRAWL MAPILLARY FOR ONE MRT
# ============================================================

def crawl_station_mapillary(
    station
):

    path = station_route_path(
        station
    )


    route = load_json(
        path,
        None
    )


    if not route:

        return None


    if (
        route.get(
            "routeStatus"
        )
        !=
        "complete"
    ):

        return route


    # --------------------------------------------------------
    # RESUME
    # --------------------------------------------------------

    if (
        route.get(
            "mapillaryStatus"
        )
        ==
        "complete"

        and
        not FORCE_RECRAWL_MAPILLARY
    ):

        return route


    checkpoints = (
        route.get(
            "checkpoints"
        )
        or
        []
    )


    if not checkpoints:

        route[
            "mapillaryStatus"
        ] = "failed"

        route[
            "mapillaryFailureReason"
        ] = "No checkpoints"


        save_json(
            path,
            route
        )


        return route


    # ========================================================
    # PASS 1 — CLOSE TO WALKING ROUTE
    # ========================================================

    candidates = (
        fetch_candidates_for_route(

            checkpoints,

            buffer_m=
                MAPILLARY_BBOX_BUFFER_M

        )
    )


    assign_images_to_checkpoints(

        checkpoints,

        candidates,

        NORMAL_IMAGE_DISTANCE_M

    )


    unique_assigned = {

        candidate["id"]

        for checkpoint
        in checkpoints

        for candidate
        in checkpoint.get(
            "imageCandidates",
            []
        )

    }


    imagery_mode = (
        "route"
    )


    # ========================================================
    # PASS 2 — EXPANDED ROUTE BUFFER
    # ========================================================

    if not unique_assigned:

        print(
            station,
            "→ expanding Mapillary search"
        )


        candidates = (
            fetch_candidates_for_route(

                checkpoints,

                buffer_m=350

            )
        )


        assign_images_to_checkpoints(

            checkpoints,

            candidates,

            EXPANDED_IMAGE_DISTANCE_M

        )


        unique_assigned = {

            candidate["id"]

            for checkpoint
            in checkpoints

            for candidate
            in checkpoint.get(
                "imageCandidates",
                []
            )

        }


        imagery_mode = (
            "expanded-route"
        )


    # ========================================================
    # PASS 3 — MRT-AREA FALLBACK
    # ========================================================

    if not unique_assigned:

        origin = (
            route[
                "origin"
            ]
        )


        fallback_point = [{

            "lat":
                origin["lat"],

            "lon":
                origin["lon"]

        }]


        bbox = bbox_for_points(

            fallback_point,

            500

        )


        fallback_raw = (
            query_mapillary_bbox(
                bbox
            )
        )


        fallback_candidates = []


        for record in fallback_raw:

            parsed = (
                parse_mapillary_record(
                    record
                )
            )


            if parsed:

                fallback_candidates.append(
                    parsed
                )


        if fallback_candidates:

            # Assign them to the first checkpoint.
            # Flag clearly as station-area fallback.

            first_checkpoint = (
                checkpoints[0]
            )


            temporary = [{

                **first_checkpoint,

                "imageCandidates":
                    []

            }]


            assign_images_to_checkpoints(

                temporary,

                fallback_candidates,

                500

            )


            first_checkpoint[
                "imageCandidates"
            ] = (
                temporary[0][
                    "imageCandidates"
                ]
            )


            unique_assigned = {

                candidate["id"]

                for candidate
                in first_checkpoint[
                    "imageCandidates"
                ]

            }


            imagery_mode = (
                "station-area-fallback"
            )


    # ========================================================
    # COUNTS
    # ========================================================

    all_references = [

        candidate

        for checkpoint
        in checkpoints

        for candidate
        in checkpoint.get(
            "imageCandidates",
            []
        )

    ]


    unique_ids = {

        candidate["id"]

        for candidate
        in all_references

    }


    route[
        "checkpoints"
    ] = checkpoints


    route[
        "mapillaryStatus"
    ] = "complete"


    route[
        "imageryMode"
    ] = imagery_mode


    route[
        "mapillaryImageCount"
    ] = len(
        all_references
    )


    route[
        "uniqueMapillaryImageCount"
    ] = len(
        unique_ids
    )


    route[
        "mapillaryCrawledAt"
    ] = time.time()


    save_json(
        path,
        route
    )


    return route

In [60]:
# ============================================================
# MASTER MAPILLARY CRAWL
# ============================================================

print()
print("==========================================")
print("CRAWLING MAPILLARY FOR ALL MRT ROUTES")
print("==========================================")
print()


crawl_results = {}


for index, station in enumerate(
    ALL_MRT_STATIONS,
    start=1
):

    print()
    print(
        f"[{index}/{len(ALL_MRT_STATIONS)}]",
        station
    )


    try:

        route = (
            crawl_station_mapillary(
                station
            )
        )


        crawl_results[
            station
        ] = route


        if not route:

            print(
                "  → route file missing"
            )

            continue


        print(
            "  route:",
            route.get(
                "routingMode"
            )
        )


        print(
            "  imagery:",
            route.get(
                "imageryMode"
            )
        )


        print(
            "  checkpoints:",
            len(
                route.get(
                    "checkpoints",
                    []
                )
            )
        )


        print(
            "  unique images:",
            route.get(
                "uniqueMapillaryImageCount",
                0
            )
        )


    except Exception as error:

        print(
            "  ERROR:",
            error
        )


        crawl_results[
            station
        ] = {

            "station":
                station,

            "mapillaryStatus":
                "failed",

            "failureReason":
                str(
                    error
                )

        }


        # Continue to next MRT.
        # Do not destroy progress.

        time.sleep(
            1
        )


CRAWLING MAPILLARY FOR ALL MRT ROUTES


[1/73] ADMIRALTY MRT
  route: walking
  imagery: route
  checkpoints: 16
  unique images: 8

[2/73] ALJUNIED MRT
  route: walking
  imagery: route
  checkpoints: 2
  unique images: 4

[3/73] ANG MO KIO MRT
  route: walking
  imagery: route
  checkpoints: 19
  unique images: 10

[4/73] BEAUTY WORLD MRT
  route: walking
  imagery: route
  checkpoints: 4
  unique images: 6

[5/73] BEDOK MRT
  route: walking
  imagery: route
  checkpoints: 19
  unique images: 34

[6/73] BEDOK NORTH MRT
  route: walking
  imagery: route
  checkpoints: 10
  unique images: 13

[7/73] BEDOK RESERVOIR MRT
  route: walking
  imagery: route
  checkpoints: 5
  unique images: 10

[8/73] BISHAN MRT
  route: walking
  imagery: route
  checkpoints: 10
  unique images: 2

[9/73] BOON KENG MRT
  route: walking
  imagery: route
  checkpoints: 17
  unique images: 26

[10/73] BOON LAY MRT
  route: walking
  imagery: route
  checkpoints: 2
  unique images: 3

[11/73] BOTANIC GARDENS 

In [61]:
# ============================================================
# BUILD MASTER ROUTE LIBRARY
# ============================================================

master_routes = []


failed_stations = []


for station in ALL_MRT_STATIONS:

    path = (
        station_route_path(
            station
        )
    )


    route = load_json(
        path,
        None
    )


    if not route:

        failed_stations.append({

            "station":
                station,

            "reason":
                "Route file not found"

        })


        continue


    master_routes.append(
        route
    )


    if (
        route.get(
            "routeStatus"
        )
        !=
        "complete"
    ):

        failed_stations.append({

            "station":
                station,

            "reason":

                route.get(
                    "failureReason"
                )

                or

                "Route failed"

        })


master_library = {

    "version":
        1,

    "pipelineVersion":
        PIPELINE_VERSION,

    "generatedAt":
        time.time(),

    "mrtCount":
        len(
            ALL_MRT_STATIONS
        ),

    "routeCount":
        len(
            master_routes
        ),

    "settings": {

        "checkpointSpacingM":
            CHECKPOINT_SPACING_M,

        "imagesPerCheckpoint":
            IMAGES_PER_CHECKPOINT,

        "normalImageDistanceM":
            NORMAL_IMAGE_DISTANCE_M,

        "expandedImageDistanceM":
            EXPANDED_IMAGE_DISTANCE_M

    },

    "routes":
        master_routes,

    "failedStations":
        failed_stations

}


save_json(

    MASTER_ROUTE_LIBRARY_PATH,

    master_library

)


save_json(

    ROUTE_FAILURE_PATH,

    failed_stations

)


print()
print(
    "Master route library:"
)

print(
    MASTER_ROUTE_LIBRARY_PATH
)


Master route library:
C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\mrt_route_library.json


In [62]:
# ============================================================
# FINAL COVERAGE REPORT
# ============================================================

coverage_rows = []


for station in ALL_MRT_STATIONS:

    route = load_json(

        station_route_path(
            station
        ),

        {}

    )


    checkpoints = (
        route.get(
            "checkpoints"
        )
        or
        []
    )


    image_refs = [

        candidate

        for checkpoint
        in checkpoints

        for candidate
        in checkpoint.get(
            "imageCandidates",
            []
        )

    ]


    unique_images = {

        candidate["id"]

        for candidate
        in image_refs

        if candidate.get(
            "id"
        )

    }


    coverage_rows.append({

        "station":
            station,

        "route_status":
            route.get(
                "routeStatus"
            ),

        "routing_mode":
            route.get(
                "routingMode"
            ),

        "imagery_mode":
            route.get(
                "imageryMode"
            ),

        "route_distance_m":
            route.get(
                "distance"
            ),

        "checkpoints":
            len(
                checkpoints
            ),

        "image_references":
            len(
                image_refs
            ),

        "unique_mapillary_images":
            len(
                unique_images
            ),

        "destination":
            (
                route.get(
                    "toilet",
                    {}
                )
                .get(
                    "name"
                )
            )

    })


coverage_df = pd.DataFrame(
    coverage_rows
)


coverage_df.to_csv(

    ROUTE_COVERAGE_CSV_PATH,

    index=False

)


display(
    coverage_df
)


print()
print("==========================================")
print("ALL-MRT ROUTE PREPROCESSING COMPLETE")
print("==========================================")
print()


print(
    "MRTs:",
    len(
        coverage_df
    )
)


print(
    "Routes completed:",
    (
        coverage_df[
            "route_status"
        ]
        ==
        "complete"
    ).sum()
)


print(
    "MRTs with Mapillary imagery:",
    (
        coverage_df[
            "unique_mapillary_images"
        ]
        >
        0
    ).sum()
)


print(
    "Total unique-image references by MRT:",
    coverage_df[
        "unique_mapillary_images"
    ].sum()
)


print()
print(
    "Master file:"
)

print(
    MASTER_ROUTE_LIBRARY_PATH
)

,station,route_status,routing_mode,imagery_mode,route_distance_m,checkpoints,image_references,unique_mapillary_images,destination
0,ADMIRALTY MRT,complete,walking,route,1153.7,16,11,8,Public toilet
1,ALJUNIED MRT,complete,walking,route,23.2,2,4,4,Public toilet
2,ANG MO KIO MRT,complete,walking,route,1364.3,19,18,10,Public toilet
3,BEAUTY WORLD MRT,complete,walking,route,177.0,4,8,6,Public toilet
4,BEDOK MRT,complete,walking,route,1392.6,19,36,34,Public toilet
...,...,...,...,...,...,...,...,...,...
68,UPPER CHANGI MRT,complete,walking,route,714.2,10,20,20,Public toilet
69,UPPER THOMSON MRT,complete,walking,route,229.3,4,8,8,Public toilet
70,WOODLANDS SOUTH MRT,complete,walking,route,269.0,5,4,2,Public toilet
71,YEW TEE MRT,complete,walking,route,805.8,12,16,14,Public toilet



ALL-MRT ROUTE PREPROCESSING COMPLETE

MRTs: 73
Routes completed: 73
MRTs with Mapillary imagery: 73
Total unique-image references by MRT: 839

Master file:
C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\mrt_route_library.json


In [63]:
# ============================================================
# GENUINE MAPILLARY COVERAGE GAPS
# ============================================================

no_imagery_df = (

    coverage_df[

        coverage_df[
            "unique_mapillary_images"
        ]
        ==
        0

    ]

    .copy()

)


print(
    "MRTs still having zero imagery after full crawl:",
    len(
        no_imagery_df
    )
)


display(
    no_imagery_df
)

MRTs still having zero imagery after full crawl: 0


,station,route_status,routing_mode,imagery_mode,route_distance_m,checkpoints,image_references,unique_mapillary_images,destination
